In [ ]:
# 1. Check GPU
import torch
print("="*60)
print("GPU CHECK")
print("="*60)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU: {gpu_name}")
    print(f"   Memory: {total_mem:.1f} GB")
    if total_mem < 20:
        print("⚠️  Warning: <20GB VRAM. L4 or A100 recommended for 3 servers.")
else:
    raise RuntimeError("❌ No GPU! Enable in Runtime > Change runtime type")

In [ ]:
# 2. Setup Directories
import os

# Clear PYTHONPATH to avoid conflicts
os.environ.pop('PYTHONPATH', None)

# Create directories
for d in ["/content/Evo-1/checkpoints/libero", 
          "/content/Evo-1/checkpoints/metaworld",
          "/content/logs"]:
    os.makedirs(d, exist_ok=True)

print("✅ Directories created")

In [ ]:
# 3. Install System Dependencies
!apt-get update -qq
!apt-get install -y -qq git wget libosmesa6-dev patchelf ffmpeg libgl1-mesa-glx
print("✅ System dependencies installed")

In [ ]:
# 4. Install Python Packages
# CRITICAL: robosuite==1.4.1 and numpy<2.0 for LIBERO compatibility

print("📦 Installing packages...\n")

# PyTorch 2.5.1 + CUDA 12.1
print("Step 1/5: PyTorch...")
!pip install -q torch==2.5.1+cu121 torchvision==0.20.1+cu121 torchaudio==2.5.1+cu121 --index-url https://download.pytorch.org/whl/cu121

# Evo-1 server dependencies
print("Step 2/5: Evo-1 dependencies...")
!pip install -q transformers>=4.45.0 timm>=1.0.0 accelerate>=1.0.0 diffusers>=0.31.0
!pip install -q websockets pyyaml opencv-python pillow "numpy>=1.26.4,<2.0" fvcore
!pip install -q einops thop cloudpickle easydict hydra-core huggingface_hub packaging ninja

# LIBERO (MUST use robosuite 1.4.1)
print("Step 3/5: LIBERO dependencies...")
!pip install -q mujoco imageio h5py bddl
!pip install -q robosuite==1.4.1  # NOT newer versions!

# MetaWorld
print("Step 4/5: MetaWorld dependencies...")
!pip install -q metaworld gymnasium

# VLABench
print("Step 5/5: VLABench dependencies...")
!pip install -q --no-cache-dir mediapy websocket-client colorlog colorama scipy tqdm
!pip install -q --no-cache-dir open3d || pip install -q open3d --upgrade

# Pre-configure robosuite (avoid interactive prompt)
import os, yaml
os.makedirs(os.path.expanduser("~/.robosuite"), exist_ok=True)
macros_file = os.path.expanduser("~/.robosuite/macros_private.py")
if not os.path.exists(macros_file):
    with open(macros_file, 'w') as f:
        f.write('import robosuite.macros as macros\nmacros.SIMULATION_TIMESTEP = 0.002\nmacros.SHOW_WARNINGS = False\n')

# Pre-configure LIBERO (avoid interactive prompt)
os.makedirs(os.path.expanduser("~/.libero"), exist_ok=True)
libero_cfg = os.path.expanduser("~/.libero/config.yaml")
if not os.path.exists(libero_cfg):
    cfg = {"benchmark_root": "/content/LIBERO/libero/libero", "bddl_files": "/content/LIBERO/libero/libero/bddl_files",
           "init_states": "/content/LIBERO/libero/libero/init_files", "datasets": "/content/LIBERO/datasets",
           "assets": "/content/LIBERO/libero/libero/assets"}
    with open(libero_cfg, 'w') as f: yaml.dump(cfg, f)

print("\n" + "="*60)
print("✅ All packages installed!")
print("   Note: robosuite==1.4.1, numpy<2.0 (LIBERO requirements)")
print("="*60)
print("\n➡️  Run Cell 5 to install flash-attn")

In [ ]:
# 5. Install flash-attn (takes 10-15 minutes)
print("🔧 Installing flash-attn (10-15 min)...")
!pip install flash-attn --no-build-isolation 2>&1 | tail -5

print("\n" + "="*60)
print("⚠️  RESTART RUNTIME NOW!")
print("="*60)
print("Go to: Runtime > Restart runtime")
print("Then continue from Cell 6")

In [ ]:
# 6. Clone Repositories (run after restart)
from pathlib import Path
import torch

# Verify flash-attn
print(f"PyTorch: {torch.__version__}")
try:
    import flash_attn
    print(f"flash-attn: {flash_attn.__version__} ✅")
except ImportError:
    raise ImportError("❌ flash-attn not found. Run Cells 4-5, restart, then continue.")

# Clone Evo-1
if not Path("/content/Evo-1/Evo_1").exists():
    print("\n📦 Cloning Evo-1...")
    !git clone https://github.com/MINT-SJTU/Evo-1.git /content/Evo-1-temp
    !cp -r /content/Evo-1-temp/* /content/Evo-1/ && rm -rf /content/Evo-1-temp
    !pip install -q -r /content/Evo-1/Evo_1/requirements.txt
else:
    print("✅ Evo-1 exists")

# Clone LIBERO
if not Path("/content/LIBERO/.git").exists():
    print("\n📦 Cloning LIBERO...")
    !git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git /content/LIBERO
    !cd /content/LIBERO && pip install -q -e .
else:
    print("✅ LIBERO exists")

# Clone rrt-algorithms (VLABench dependency)
if not Path("/content/rrt-algorithms/.git").exists():
    print("\n📦 Cloning rrt-algorithms...")
    !git clone https://github.com/motion-planning/rrt-algorithms.git /content/rrt-algorithms
    !cd /content/rrt-algorithms && pip install -q -e .
else:
    print("✅ rrt-algorithms exists")

# Clone VLABench
if not Path("/content/VLABench/.git").exists():
    print("\n📦 Cloning VLABench...")
    !git clone https://github.com/OpenMOSS/VLABench.git /content/VLABench
    !cd /content/VLABench && pip install -q -e .
else:
    print("✅ VLABench exists")
    !cd /content/VLABench && pip install -q -e .

print("\n✅ All repositories ready")

In [ ]:
# 7. Setup VLABench Assets from Google Drive
# You must upload obj.zip and scenes.zip to your Drive first

from pathlib import Path
from google.colab import drive

# Mount Drive
if not Path("/content/drive").exists():
    drive.mount('/content/drive')

# Configure your Drive path here
DRIVE_PATH = "/content/drive/MyDrive/Research/URECA/VLABench"  # <-- Edit if needed

assets_dir = Path("/content/VLABench/VLABench/assets")
assets_dir.mkdir(parents=True, exist_ok=True)

# Extract obj.zip
obj_zip = f"{DRIVE_PATH}/obj.zip"
if Path(obj_zip).exists() and not (assets_dir / "obj").exists():
    print("📂 Extracting obj.zip...")
    !unzip -q -o "{obj_zip}" -d {assets_dir}
    print("✅ obj.zip extracted")
elif (assets_dir / "obj").exists():
    print("✅ obj/ already exists")
else:
    print(f"❌ obj.zip not found at {obj_zip}")

# Extract scenes.zip
scenes_zip = f"{DRIVE_PATH}/scenes.zip"
if Path(scenes_zip).exists() and not (assets_dir / "scenes").exists():
    print("📂 Extracting scenes.zip...")
    !unzip -q -o "{scenes_zip}" -d {assets_dir}
    print("✅ scenes.zip extracted")
elif (assets_dir / "scenes").exists():
    print("✅ scenes/ already exists")
else:
    print(f"❌ scenes.zip not found at {scenes_zip}")

print("\n✅ VLABench assets setup complete")

In [ ]:
# 8. Download Checkpoints
from huggingface_hub import snapshot_download, hf_hub_download
from pathlib import Path
import os

CHECKPOINTS = {
    "libero": {"repo": "MINT-SJTU/Evo1_LIBERO", "dir": "/content/Evo-1/checkpoints/libero"},
    "metaworld": {"repo": "MINT-SJTU/Evo1_MetaWorld", "dir": "/content/Evo-1/checkpoints/metaworld"}
}

# The actual model weights file we need
MODEL_FILE = "mp_rank_00_model_states.pt"

print("📥 Downloading checkpoints...")
print("   (This may take 5-10 minutes per checkpoint)\n")

for name, cfg in CHECKPOINTS.items():
    ckpt_dir = Path(cfg["dir"])
    model_path = ckpt_dir / MODEL_FILE
    
    # Check if the actual model weights exist (not just config files)
    if model_path.exists() and model_path.stat().st_size > 1_000_000:  # >1MB means real weights
        print(f"✅ {name}: model weights exist ({model_path.stat().st_size / 1e9:.2f} GB)")
    else:
        print(f"⏳ Downloading {name} checkpoint...")
        
        # Remove incomplete download if exists
        if ckpt_dir.exists():
            import shutil
            shutil.rmtree(ckpt_dir)
        
        # Download with explicit settings
        try:
            snapshot_download(
                repo_id=cfg["repo"],
                local_dir=cfg["dir"],
                local_dir_use_symlinks=False,
                resume_download=True,
            )
            
            # Verify download
            if model_path.exists() and model_path.stat().st_size > 1_000_000:
                print(f"✅ {name}: downloaded ({model_path.stat().st_size / 1e9:.2f} GB)")
            else:
                print(f"⚠️  {name}: download incomplete, trying direct file download...")
                # Try downloading the model file directly
                hf_hub_download(
                    repo_id=cfg["repo"],
                    filename=MODEL_FILE,
                    local_dir=cfg["dir"],
                    local_dir_use_symlinks=False,
                )
                print(f"✅ {name}: downloaded via direct method")
        except Exception as e:
            print(f"❌ {name} failed: {e}")
            print("   Try manually: huggingface-cli download {cfg['repo']} --local-dir {cfg['dir']}")

# Final verification
print("\n" + "="*60)
print("Checkpoint Verification:")
for name, cfg in CHECKPOINTS.items():
    model_path = Path(cfg["dir"]) / MODEL_FILE
    if model_path.exists():
        size_gb = model_path.stat().st_size / 1e9
        status = "✅" if size_gb > 0.1 else "⚠️ too small"
        print(f"  {name}: {size_gb:.2f} GB {status}")
    else:
        print(f"  {name}: ❌ MISSING {MODEL_FILE}")

print("\n✅ Checkpoints ready (VLABench uses LIBERO checkpoint zero-shot)")

In [ ]:
# 9. Verify Dependencies
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

for pkg, name in [("flash_attn", "flash-attn"), ("transformers", "transformers"), 
                  ("mujoco", "mujoco"), ("robosuite", "robosuite")]:
    try:
        m = __import__(pkg)
        print(f"{name}: {getattr(m, '__version__', 'OK')} ✅")
    except ImportError:
        print(f"{name}: ❌")

print("\n✅ Ready to run benchmarks!")

In [ ]:
# 10. Create Client Scripts (with server fix for persistent connections + large message support)
import re

print("📝 Creating/patching client scripts...")

# 100MB max message size for large image observations
MAX_MSG_SIZE = 100 * 1024 * 1024  # 100MB

# Create a completely new multi-server script with proper persistent connection handling
# Based on the original Evo1_server.py - uses load_model_and_normalizer pattern
multi_server = '''#!/usr/bin/env python3
"""
Evo-1 Multi-Server with Persistent WebSocket Connections
Handles multiple requests per connection for MetaWorld/VLABench clients
Based on original Evo1_server.py model loading pattern
"""
import sys
import os
import asyncio
import json
import numpy as np
import torch
from PIL import Image
import websockets
import argparse

# 100MB max message size for large image observations
MAX_SIZE = 100 * 1024 * 1024

# Parse args first
parser = argparse.ArgumentParser()
parser.add_argument("--port", type=int, required=True)
parser.add_argument("--checkpoint", type=str, required=True)
parser.add_argument("--name", type=str, default="evo1")
args = parser.parse_args()

print(f"[{args.name}] Starting on port {args.port}")
print(f"[{args.name}] Loading checkpoint from {args.checkpoint}")

# CRITICAL: Set up Python path correctly - must be parent of scripts/
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
print(f"[{args.name}] Python path: {sys.path[:3]}")

# Import model using the SAME pattern as original Evo1_server.py
from scripts.Evo1 import EVO1

# ============ Normalizer class from original server ============
class Normalizer:
    def __init__(self, stats_or_path):
        if isinstance(stats_or_path, str):
            with open(stats_or_path, "r") as f:
                stats = json.load(f)
        else:
            stats = stats_or_path

        def pad_to_24(x):
            x = torch.tensor(x, dtype=torch.float32)
            if x.shape[0] < 24:
                pad = torch.zeros(24 - x.shape[0], dtype=torch.float32)
                x = torch.cat([x, pad], dim=0)
            elif x.shape[0] > 24:
                raise ValueError(f"Input length {x.shape[0]} exceeds expected 24")
            return x

        if len(stats) != 1:
            raise ValueError(f"norm_stats.json should contain only one robot key, but: {list(stats.keys())}")

        robot_key = list(stats.keys())[0]
        robot_stats = stats[robot_key]

        self.state_min = pad_to_24(robot_stats["observation.state"]["min"])
        self.state_max = pad_to_24(robot_stats["observation.state"]["max"])
        self.action_min = pad_to_24(robot_stats["action"]["min"])
        self.action_max = pad_to_24(robot_stats["action"]["max"])

    def normalize_state(self, state: torch.Tensor) -> torch.Tensor:
        state_min = self.state_min.to(state.device, dtype=state.dtype)
        state_max = self.state_max.to(state.device, dtype=state.dtype)
        return torch.clamp(2 * (state - state_min) / (state_max - state_min + 1e-8) - 1, -1.0, 1.0)

    def denormalize_action(self, action: torch.Tensor) -> torch.Tensor:
        action_min = self.action_min.to(action.device, dtype=action.dtype)
        action_max = self.action_max.to(action.device, dtype=action.dtype)
        if action.ndim == 1:
            action = action.view(1, -1)
        return (action + 1.0) / 2.0 * (action_max - action_min + 1e-8) + action_min


def load_model_and_normalizer(ckpt_dir):
    """Load model exactly as original Evo1_server.py does"""
    config = json.load(open(os.path.join(ckpt_dir, "config.json")))
    stats = json.load(open(os.path.join(ckpt_dir, "norm_stats.json")))

    config["finetune_vlm"] = False
    config["finetune_action_head"] = False
    config["num_inference_timesteps"] = 32

    model = EVO1(config).eval()
    ckpt_path = os.path.join(ckpt_dir, "mp_rank_00_model_states.pt")

    # weights_only=False is required for this checkpoint format (contains custom objects)
    checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    model.load_state_dict(checkpoint["module"], strict=True)
    device = config.get("device", "cuda")
    model = model.to(device)

    normalizer = Normalizer(stats)
    return model, normalizer, config


# Load model using the original pattern
print("Loading EVO1 model...")
model, normalizer, config = load_model_and_normalizer(args.checkpoint)
print(f"num_inference_timesteps: {config.get('num_inference_timesteps', 32)}")
print(f"EVO1 model loaded successfully!")


async def handle_client(websocket):
    """Handle a single client with persistent connection - multiple requests allowed"""
    client_id = id(websocket)
    print(f"Client {client_id} connected")
    
    try:
        async for message in websocket:
            try:
                # Parse the incoming message
                data = json.loads(message)
                print(f"Received JSON observation from client {client_id}")
                
                # Process images
                images = []
                for img_data in data.get("image", []):
                    img_array = np.array(img_data, dtype=np.uint8)
                    if img_array.ndim == 3:
                        img = Image.fromarray(img_array)
                        images.append(img)
                
                # Pad to 3 images if needed
                while len(images) < 3:
                    images.append(Image.new('RGB', (448, 448), (0, 0, 0)))
                
                # Get state and prompt
                state = np.array(data.get("state", [0.0] * 7), dtype=np.float32)
                prompt = data.get("prompt", "")
                image_mask = data.get("image_mask", [1, 1, 0])
                action_mask = data.get("action_mask", [1]*7 + [0]*17)
                
                # Convert to tensors
                image_mask_tensor = torch.tensor(image_mask, dtype=torch.int32).cuda()
                action_mask_tensor = torch.tensor([action_mask], dtype=torch.int32).cuda()
                
                # Normalize state
                state_tensor = torch.tensor(state, dtype=torch.float32).cuda()
                # Pad state to 24 dims if needed
                if state_tensor.shape[0] < 24:
                    state_tensor = torch.cat([state_tensor, torch.zeros(24 - state_tensor.shape[0], device="cuda")])
                state_normalized = normalizer.normalize_state(state_tensor.unsqueeze(0))
                
                print(f"image_mask: {image_mask_tensor}")
                print(f"action_mask shape: {action_mask_tensor.shape}")
                
                # Get VL embeddings and predict actions
                with torch.no_grad():
                    fused_tokens = model.get_vl_embeddings(
                        images=images,
                        image_mask=image_mask_tensor,
                        prompt=prompt
                    )
                    
                    # Predict actions - returns [batch, horizon*action_dim] = [1, 1200]
                    actions = model.predict_action(
                        fused_tokens=fused_tokens,
                        state=state_normalized,
                        action_mask=action_mask_tensor
                    )
                    print(f"Raw action shape: {actions.shape}")
                    
                    # Reshape from [1, 1200] to [1, 50, 24]
                    if actions.ndim == 2 and actions.shape[1] == 1200:
                        actions = actions.view(1, 50, 24)
                    print(f"Reshaped action shape: {actions.shape}")
                    
                    # Denormalize each of the 50 timesteps individually
                    # normalizer.denormalize_action expects [action_dim] or [batch, action_dim]
                    horizon = actions.shape[1]
                    actions_denorm = []
                    for t in range(horizon):
                        action_t = actions[0, t, :]  # [24]
                        action_t_denorm = normalizer.denormalize_action(action_t)  # [1, 24]
                        actions_denorm.append(action_t_denorm.squeeze(0))  # [24]
                    actions = torch.stack(actions_denorm, dim=0)  # [50, 24]
                    print(f"Denormalized action shape: {actions.shape}")
                
                # Convert to numpy and send
                actions_np = actions.cpu().numpy()  # [50, 24]
                actions_list = actions_np.tolist()
                
                await websocket.send(json.dumps(actions_list))
                print(f"Sent action chunk to client {client_id} (shape: {actions_np.shape})")
                
            except json.JSONDecodeError as e:
                print(f"JSON decode error: {e}")
                await websocket.send(json.dumps({"error": "Invalid JSON"}))
            except Exception as e:
                print(f"Error processing request: {e}")
                import traceback
                traceback.print_exc()
                await websocket.send(json.dumps({"error": str(e)}))
                
    except websockets.exceptions.ConnectionClosed:
        print(f"Client {client_id} disconnected normally")
    except Exception as e:
        print(f"Client {client_id} error: {e}")
    finally:
        print(f"Client {client_id} connection closed")


async def main():
    port = args.port
    print(f"EVO1 server running at ws://0.0.0.0:{port}")
    print(f"Max message size: {MAX_SIZE / 1024 / 1024:.0f} MB")
    
    async with websockets.serve(
        handle_client, 
        "0.0.0.0", 
        port,
        max_size=MAX_SIZE,           # CRITICAL: Allow large messages (100MB)
        ping_interval=30,            # Keep connection alive
        ping_timeout=60,
        close_timeout=10
    ):
        await asyncio.Future()  # Run forever

if __name__ == "__main__":
    asyncio.run(main())
'''

with open("/content/Evo-1/Evo_1/scripts/Evo1_multi_server.py", 'w') as f:
    f.write(multi_server)
print("✅ Multi-server script created (with 100MB message limit)")

# --- Patch LIBERO client to increase max message size ---
libero_client = "/content/Evo-1/LIBERO_evaluation/libero_client_4tasks.py"
with open(libero_client, 'r') as f:
    content = f.read()

# Fix SERVER_URL
content = re.sub(r'SERVER_URL\s*=\s*["\'][^"\']*["\']', 'SERVER_URL = "ws://localhost:9001"', content)

# Fix MUJOCO_GL
content = content.replace('os.environ["MUJOCO_GL"] = "osmesa"', 'os.environ["MUJOCO_GL"] = "egl"')
content = content.replace('os.environ["MUJOCO_GL"] = "osmes"', 'os.environ["MUJOCO_GL"] = "egl"')

# CRITICAL: Add max_size to websockets.connect() - the key fix!
# Find the websockets.connect line and add max_size parameter
if 'max_size=' not in content:
    # Pattern: async with websockets.connect(SERVER_URL) as ws:
    content = re.sub(
        r'async with websockets\.connect\(SERVER_URL\)',
        'async with websockets.connect(SERVER_URL, max_size=100*1024*1024)',
        content
    )
    # Also handle if it's a variable
    content = re.sub(
        r'async with websockets\.connect\(([^,\)]+)\) as',
        r'async with websockets.connect(\1, max_size=100*1024*1024) as',
        content
    )

with open(libero_client, 'w') as f:
    f.write(content)
print("✅ LIBERO client patched (localhost:9001, max_size=100MB)")

# --- Patch MetaWorld client ---
mw_client = "/content/Evo-1/MetaWorld_evaluation/mt50_evo1_client_prompt.py"
with open(mw_client, 'r') as f:
    content = f.read()

content = re.sub(r'SERVER_URL\s*=\s*["\'][^"\']*["\']', 'SERVER_URL = "ws://localhost:9002"', content)
content = re.sub(r'SHOW_WINDOW\s*=\s*True', 'SHOW_WINDOW = False', content)
content = re.sub(r'SAVE_VIDEO\s*=\s*True', 'SAVE_VIDEO = False', content)

# Add max_size to websockets.connect if not present
if 'max_size=' not in content:
    content = re.sub(
        r'async with websockets\.connect\(([^,\)]+)\) as',
        r'async with websockets.connect(\1, max_size=100*1024*1024) as',
        content
    )

with open(mw_client, 'w') as f:
    f.write(content)
print("✅ MetaWorld client patched (localhost:9002, max_size=100MB)")

# --- Create VLABench client using ASYNC websockets (same as LIBERO/MetaWorld) ---
vlabench_client = '''#!/usr/bin/env python3
"""VLABench WebSocket Client for Evo-1 - Using async websockets library"""
import asyncio
import argparse
import json
import math
import os
import sys
import numpy as np
import websockets

os.environ["MUJOCO_GL"] = "egl"
sys.path.insert(0, "/content/VLABench")

# Import at module level to trigger registration
from VLABench.robots import *
from VLABench.tasks import *
from VLABench.envs import load_env
from VLABench.utils.register import register

# 100MB max message size
MAX_SIZE = 100 * 1024 * 1024

def encode_image(img):
    return img.astype(np.uint8).tolist()

def quat2axisangle(q):
    q = np.array(q, dtype=np.float64)
    q[3] = max(-1, min(1, q[3]))
    den = np.sqrt(1 - q[3]**2)
    return np.zeros(3) if abs(den) < 1e-8 else (q[:3] * 2 * math.acos(q[3])) / den

def prepare_observation(obs, instruction):
    """Prepare observation dict for sending to server"""
    import cv2
    
    # Get RGB image
    img = obs.get("rgb", np.zeros((448, 448, 3), dtype=np.uint8))
    if img.ndim == 4:
        img = img[2] if img.shape[0] > 2 else img[0]
    
    # Get wrist image
    wrist = obs.get("wrist", np.zeros((448, 448, 3), dtype=np.uint8))
    if wrist.ndim == 4:
        wrist = wrist[-1]
    
    # Resize if needed
    if img.shape[:2] != (448, 448):
        img = cv2.resize(img, (448, 448))
    if wrist.shape[:2] != (448, 448):
        wrist = cv2.resize(wrist, (448, 448))
    
    # Get end-effector state
    ee = obs.get("ee_state", np.zeros(8))
    if len(ee) >= 7:
        state = np.concatenate((
            ee[:3],
            quat2axisangle(ee[3:7]),
            [ee[7]] if len(ee) > 7 else [0]
        )).tolist()
    else:
        state = [0.0] * 7
    
    data = {
        "image": [
            encode_image(img),
            encode_image(wrist),
            encode_image(np.zeros((448, 448, 3), dtype=np.uint8))
        ],
        "state": state,
        "prompt": instruction,
        "image_mask": [1, 1, 0],
        "action_mask": [1] * 7 + [0] * 17
    }
    return data

async def run_episode(ws, env, instruction, max_steps):
    """Run a single episode"""
    timestep = env.reset()
    obs = timestep.observation
    done = False
    step = 0
    
    # Get the environment's action dimension from the action spec
    action_spec = env.action_spec()
    env_action_dim = action_spec.shape[0]  # Typically 7 or 9 for VLABench
    print(f"    Environment expects {env_action_dim}-dim actions")
    
    while not done and step < max_steps:
        # Prepare and send observation
        obs_dict = {
            "rgb": obs.get("rgb", np.zeros((4, 480, 480, 3))),
            "ee_state": obs.get("ee_state", np.zeros(8)),
            "wrist": obs.get("wrist", obs.get("rgb", np.zeros((4, 480, 480, 3))))
        }
        send_data = prepare_observation(obs_dict, instruction)
        
        await ws.send(json.dumps(send_data))
        print(f"    [Step {step}] Sent observation")
        
        # Receive action
        result = await ws.recv()
        try:
            action_list = json.loads(result)
            # Check if server returned an error
            if isinstance(action_list, dict) and "error" in action_list:
                print(f"    ❌ Server error: {action_list['error']}")
                break
            actions = np.array(action_list)  # [50, 24] - 50 timesteps, 24 dims each
            action_full = actions[0]  # First timestep action [24]
            # Slice to match environment's expected action dimension
            action = action_full[:env_action_dim]
            print(f"    [Step {step}] Received action, using first {env_action_dim} dims of {action_full.shape}")
        except Exception as e:
            print(f"    ❌ Action parsing failed: {e}")
            break
        
        # Execute action
        timestep = env.step(action)
        obs = timestep.observation
        done = timestep.last()
        step += 1
    
    # VLABench uses different success check methods depending on task
    # Try multiple method names for compatibility
    success = False
    if hasattr(env, 'task'):
        if hasattr(env.task, 'check_success'):
            success = env.task.check_success(env.physics)
        elif hasattr(env.task, 'is_success'):
            success = env.task.is_success()
        elif hasattr(env.task, 'success'):
            success = env.task.success
        elif hasattr(env.task, '_check_success'):
            success = env.task._check_success(env.physics)
    # Also check env-level success
    if not success and hasattr(env, 'is_success'):
        success = env.is_success()
    if not success and hasattr(timestep, 'reward'):
        success = timestep.reward > 0.5  # Fallback: high reward = success
    
    return success, step

async def evaluate_task(server_url, task_name, n_episodes, max_steps):
    """Evaluate a single task"""
    results = {"task": task_name, "episodes": [], "success_rate": 0}
    successes = 0
    
    for ep in range(n_episodes):
        print(f"  Episode {ep + 1}/{n_episodes}")
        try:
            # Create environment
            env = load_env(task_name, random_init=True, run_mode="eval", eval=False)
            instruction = env.task.get_instruction()
            print(f"    Instruction: {instruction[:60]}...")
            
            # Connect to server for this episode
            async with websockets.connect(server_url, max_size=MAX_SIZE) as ws:
                success, steps = await run_episode(ws, env, instruction, max_steps)
            
            successes += success
            print(f"    {'✅ Success' if success else '❌ Failed'} ({steps} steps)")
            results["episodes"].append({"success": bool(success), "steps": steps})
            
        except Exception as e:
            import traceback
            print(f"    ⚠️ Error: {e}")
            traceback.print_exc()
            results["episodes"].append({"success": False, "error": str(e)})
    
    results["success_rate"] = successes / n_episodes if n_episodes > 0 else 0
    return results

async def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--host", default="localhost")
    parser.add_argument("--port", type=int, default=9003)
    parser.add_argument("--tasks", nargs="+", default=["select_toy", "select_fruit", "select_drink"])
    parser.add_argument("--n-episodes", type=int, default=5)
    parser.add_argument("--max-steps", type=int, default=200)
    parser.add_argument("--save-dir", default="/content/logs/vlabench")
    args = parser.parse_args()
    
    server_url = f"ws://{args.host}:{args.port}"
    
    print("VLABench Evaluation for Evo-1")
    print("=" * 60)
    print(f"Server: {server_url}")
    print(f"Max message size: {MAX_SIZE / 1024 / 1024:.0f} MB")
    print(f"Tasks: {args.tasks}")
    print(f"Episodes per task: {args.n_episodes}")
    print(f"✅ Loaded {len(register._tasks)} VLABench tasks")
    print("=" * 60)
    
    os.makedirs(args.save_dir, exist_ok=True)
    
    all_results = {}
    for task in args.tasks:
        print(f"\\nEvaluating: {task}")
        try:
            result = await evaluate_task(server_url, task, args.n_episodes, args.max_steps)
            all_results[task] = result
            print(f"  Success Rate: {result['success_rate'] * 100:.1f}%")
        except Exception as e:
            print(f"  ❌ Task failed: {e}")
            all_results[task] = {"error": str(e), "success_rate": 0}
    
    # Summary
    print("\\n" + "=" * 60)
    print("Results Summary")
    print("=" * 60)
    for task, r in all_results.items():
        sr = r.get("success_rate", 0) * 100
        print(f"  {task}: {sr:.1f}%")
    
    avg_sr = np.mean([r.get("success_rate", 0) for r in all_results.values()]) * 100
    print(f"\\n  Average: {avg_sr:.1f}%")
    
    # Save results
    with open(f"{args.save_dir}/results.json", "w") as f:
        json.dump(all_results, f, indent=2, default=str)
    print(f"\\nResults saved to {args.save_dir}/results.json")

if __name__ == "__main__":
    asyncio.run(main())
'''

with open("/content/vlabench_client.py", 'w') as f:
    f.write(vlabench_client)
print("✅ VLABench client created (async websockets, localhost:9003, max_size=100MB)")

print("\n✅ All client scripts ready!")
print("   Server max message size: 100MB")
print("   Client max message size: 100MB (all using async websockets)")
print("   Client max message size: 100MB")

In [ ]:
# 11. Run All Benchmarks
import subprocess, time, os, socket, signal

os.environ["MUJOCO_GL"] = "egl"
os.environ["DISPLAY"] = ""
os.makedirs("/content/logs/vlabench", exist_ok=True)

# ============ CLEANUP: Kill any existing processes on our ports ============
PORTS = [9001, 9002, 9003]
print("🧹 Cleaning up any existing servers...")
for port in PORTS:
    try:
        # Find process using this port
        result = subprocess.run(
            f"lsof -ti:{port}", shell=True, capture_output=True, text=True
        )
        if result.stdout.strip():
            pids = result.stdout.strip().split('\n')
            for pid in pids:
                try:
                    os.kill(int(pid), signal.SIGKILL)
                    print(f"  Killed process {pid} on port {port}")
                except (ProcessLookupError, ValueError):
                    pass
    except Exception as e:
        pass  # No process on this port

# Also kill any python processes running our server script
subprocess.run("pkill -f 'Evo1_multi_server.py' 2>/dev/null", shell=True)
subprocess.run("pkill -f 'Evo1_server.py' 2>/dev/null", shell=True)
time.sleep(2)  # Give OS time to release ports
print("✅ Cleanup complete\n")

# ============ Helper functions ============
def check_port_websocket(port, timeout=5):
    """Check if a WebSocket server is ready on the given port."""
    import websocket
    try:
        ws = websocket.create_connection(f"ws://localhost:{port}", timeout=timeout)
        ws.close()
        return True
    except Exception:
        return False

def wait_for_server(port, name, max_wait=180):
    """Wait until server is ready on the given port."""
    print(f"  Waiting for {name} on port {port}...", end="", flush=True)
    start = time.time()
    while time.time() - start < max_wait:
        if check_port_websocket(port):
            print(f" ✅ ready ({int(time.time()-start)}s)")
            return True
        time.sleep(5)
        print(".", end="", flush=True)
    print(f" ❌ timeout after {max_wait}s")
    return False

# ============ Server configs ============
SERVERS = [
    (9001, "/content/Evo-1/checkpoints/libero", "libero"),
    (9002, "/content/Evo-1/checkpoints/metaworld", "metaworld"),
    (9003, "/content/Evo-1/checkpoints/libero", "vlabench"),  # Zero-shot uses LIBERO checkpoint
]

processes = []
# CRITICAL: Add PYTHONUNBUFFERED to force output flushing to log files
client_env = {**os.environ, "PYTHONPATH": "/content/LIBERO:/content/VLABench", "PYTHONUNBUFFERED": "1"}

print("🚀 Starting Evo-1 servers (one at a time to manage GPU memory)...")
print("="*60)

# Start servers sequentially and wait for each to be ready
for port, ckpt, name in SERVERS:
    # CRITICAL: Run from Evo_1 directory so "from scripts.Evo1 import EVO1" works
    # Use python -u for unbuffered output
    cmd = f"cd /content/Evo-1/Evo_1 && python -u scripts/Evo1_multi_server.py --port {port} --checkpoint {ckpt} --name {name}"
    log = open(f"/content/logs/server_{name}.log", "w")
    p = subprocess.Popen(cmd, shell=True, stdout=log, stderr=log, env={**os.environ, "PYTHONPATH": "/content/Evo-1/Evo_1", "PYTHONUNBUFFERED": "1"})
    processes.append((f"server_{name}", p, log))
    print(f"  Started {name} server (port {port})")
    
    # Wait for this server to be ready before starting the next
    if not wait_for_server(port, name, max_wait=180):
        print(f"\n❌ Server {name} failed to start. Check /content/logs/server_{name}.log")
        print("Showing last 20 lines of log:")
        !tail -20 /content/logs/server_{name}.log
        raise RuntimeError(f"Server {name} failed to start")

print("\n✅ All servers ready!")
print("="*60)

print("\n🔬 Starting benchmark clients...")

# LIBERO - use python -u for unbuffered output
cmd = "cd /content/LIBERO && python -u /content/Evo-1/LIBERO_evaluation/libero_client_4tasks.py"
log = open("/content/logs/client_libero.log", "w")
p = subprocess.Popen(cmd, shell=True, stdout=log, stderr=log, stdin=subprocess.DEVNULL, env=client_env)
processes.append(("client_libero", p, log))
print("  ✅ LIBERO client started")

# MetaWorld - use python -u for unbuffered output
cmd = "cd /content/Evo-1/MetaWorld_evaluation && python -u mt50_evo1_client_prompt.py"
log = open("/content/logs/client_metaworld.log", "w")
p = subprocess.Popen(cmd, shell=True, stdout=log, stderr=log, stdin=subprocess.DEVNULL, env=client_env)
processes.append(("client_metaworld", p, log))
print("  ✅ MetaWorld client started")

# VLABench - use python -u for unbuffered output
cmd = "python -u /content/vlabench_client.py --host localhost --port 9003 --n-episodes 3"
log = open("/content/logs/client_vlabench.log", "w")
p = subprocess.Popen(cmd, shell=True, stdout=log, stderr=log, stdin=subprocess.DEVNULL, env=client_env)
processes.append(("client_vlabench", p, log))
print("  ✅ VLABench client started")

print("\n" + "="*60)
print("✅ All benchmarks running!")
print("="*60)
print("Logs: /content/logs/")
print("\n⏳ Monitoring... (Interrupt kernel to stop)")

try:
    while True:
        time.sleep(60)
        print("\n--- Status ---")
        all_done = True
        for name, p, _ in processes:
            if p.poll() is None:
                print(f"  {name}: Running")
                all_done = False
            else:
                status = "✅ Done" if p.returncode == 0 else f"❌ Failed ({p.returncode})"
                print(f"  {name}: {status}")
        
        # Show latest output from clients
        print("\nLatest output:")
        !tail -n 2 /content/logs/client_*.log 2>/dev/null | head -20
        
        if all_done:
            print("\n✅ All processes completed!")
            break
            
except KeyboardInterrupt:
    print("\n🛑 Stopping all processes...")
    for name, p, log in processes:
        p.terminate()
        log.close()
    # Also cleanup ports
    for port in PORTS:
        subprocess.run(f"lsof -ti:{port} | xargs -r kill -9 2>/dev/null", shell=True)
    print("✅ Stopped")

In [ ]:
# 12. View Results
print("📊 Benchmark Results\n")

print("--- LIBERO ---")
!tail -20 /content/logs/client_libero.log 2>/dev/null || echo "No results yet"

print("\n--- MetaWorld ---")
!tail -20 /content/logs/client_metaworld.log 2>/dev/null || echo "No results yet"

print("\n--- VLABench ---")
!tail -20 /content/logs/client_vlabench.log 2>/dev/null || echo "No results yet"

In [ ]:
# 13. Download All Logs
import zipfile
from datetime import datetime
from google.colab import files

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_path = f"/content/evo1_logs_{timestamp}.zip"

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, files_list in os.walk("/content/logs"):
        for f in files_list:
            filepath = os.path.join(root, f)
            z.write(filepath, os.path.relpath(filepath, "/content/logs"))

print(f"📦 Created {zip_path}")
files.download(zip_path)